# NB01 — Project Setup & Data Collection
## EPGT Research Pipeline | Emoji Pragmatic Graph Transformer

---

### Tujuan Notebook

Notebook ini adalah **titik awal** seluruh pipeline penelitian EPGT. Notebook ini bertanggung jawab atas dua tahap kritis:

**Tahap 1 — Automatic Project Setup**
Membuat seluruh struktur folder proyek di Google Drive secara otomatis menggunakan Python, kemudian membuat template awal untuk semua modul source code di dalam `src/`. Proses ini bersifat idempotent — aman dijalankan ulang tanpa merusak file yang sudah ada.

**Tahap 2 — Data Collection Pipeline**
Mengumpulkan data teks media sosial Indonesia yang mengandung emoji dari empat platform: Twitter/X, YouTube, TikTok, dan Instagram. Pipeline ini menerapkan inclusion filter sesuai blueprint (bahasa Indonesia, minimum 1 emoji, minimum 3 token), melakukan deduplication, dan menyimpan hasil ke `data/raw/` di Google Drive.

---

### Struktur Output
```
EPGT_Research/
├── data/raw/twitter/raw_twitter.csv
├── data/raw/youtube/raw_youtube.csv
├── data/raw/tiktok/raw_tiktok.csv
├── data/raw/instagram/raw_instagram.csv
└── src/  (semua template modul)
```

---

**Blueprint Reference:** Section 2.1 Dataset Construction Strategy, Phase 1 (Weeks 1–4)

---
## BAGIAN 1 — ENVIRONMENT SETUP
### Cell 1.1 — Install Dependencies

In [1]:
# ============================================================
# CELL 1.1 — Install Dependencies (FIXED)
# ============================================================

import subprocess, sys

def install(package: str):
    """Install package dan tampilkan status."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", package],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"  [{status}] {package}")
    if result.returncode != 0:
        print(f"         {result.stderr.strip()[:120]}")

print("Installing dependencies...")
print("(Versi tidak di-pin agar kompatibel dengan numpy Colab)\n")

# Core libraries — tanpa pin versi untuk hindari numpy conflict
packages = [
    "tweepy",
    "google-api-python-client",
    "google-auth-httplib2",
    "google-auth-oauthlib",
    "instaloader",
    "emoji",
    "langdetect",
    "python-dotenv",
    "tqdm",
    # playwright diinstall terpisah karena butuh setup browser
]

for pkg in packages:
    install(pkg)

# Playwright — install browser secara terpisah
print("\n  Installing playwright...")
!pip install -q playwright
!playwright install chromium --with-deps -q 2>/dev/null
print("  [OK] playwright + chromium")

print("\n" + "="*55)
print("Installation complete.")
print("="*55)
print("\n⚠️  WAJIB: Restart runtime sebelum lanjut!")
print("   Menu: Runtime > Restart session")
print("   Setelah restart, jalankan ulang dari Cell 1.2")
print("   (Cell 1.1 TIDAK perlu dijalankan ulang)")

Installing dependencies...
(Versi tidak di-pin agar kompatibel dengan numpy Colab)

  [OK] tweepy
  [OK] google-api-python-client
  [OK] google-auth-httplib2
  [OK] google-auth-oauthlib
  [OK] instaloader
  [OK] emoji
  [OK] langdetect
  [OK] python-dotenv
  [OK] tqdm

  Installing playwright...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 19.1 MB/s eta 0:00:00
  [OK] playwright + chromium

Installation complete.

⚠️  WAJIB: Restart runtime sebelum lanjut!
   Menu: Runtime > Restart session
   Setelah restart, jalankan ulang dari Cell 1.2
   (Cell 1.1 TIDAK perlu dijalankan ulang)


### Cell 1.2 — Mount Google Drive

In [2]:
# ============================================================
# CELL 1.2 — Mount Google Drive
# ============================================================

from google.colab import drive
import os

MOUNT_PATH = "/content/drive"

if not os.path.exists(MOUNT_PATH):
    drive.mount(MOUNT_PATH)
else:
    print("Google Drive sudah ter-mount.")
    # Force remount jika diperlukan:
    # drive.mount(MOUNT_PATH, force_remount=True)

print(f"Drive mount path : {MOUNT_PATH}")
print(f"MyDrive path     : {MOUNT_PATH}/MyDrive")

Mounted at /content/drive
Drive mount path : /content/drive
MyDrive path     : /content/drive/MyDrive


---
## BAGIAN 2 — AUTOMATIC PROJECT SETUP
### Cell 2.1 — Definisi Konfigurasi Global Proyek

In [3]:
# ============================================================
# CELL 2.1 — Project Configuration
# ============================================================

from pathlib import Path

# ── PATH CONFIGURATION ────────────────────────────────────
DRIVE_ROOT    = Path("/content/drive/MyDrive/EPGT_Research")
RUNTIME_ROOT  = Path("/content/EPGT_runtime")

# ── DATASET TARGETS (sesuai blueprint Section 2.1.1) ──────
PLATFORM_TARGETS = {
    "twitter"   : 20000,
    "youtube"   : 12500,
    "tiktok"    : 10000,
    "instagram" : 7500,
}
TOTAL_TARGET     = 50000
COLLECTION_BUFFER= 1.25    # Kumpulkan 25% lebih banyak

# ── INCLUSION CRITERIA (sesuai blueprint Section 2.1.3) ───
MIN_EMOJI_COUNT  = 1
MIN_TOKEN_COUNT  = 3
LANG_TARGET      = "id"    # Bahasa Indonesia

# ── RANDOM SEED ───────────────────────────────────────────
RANDOM_SEED = 42

print("Project Configuration:")
print(f"  Drive Root    : {DRIVE_ROOT}")
print(f"  Runtime Root  : {RUNTIME_ROOT}")
print(f"  Total Target  : {TOTAL_TARGET:,} samples")
print(f"  Platforms     : {list(PLATFORM_TARGETS.keys())}")

Project Configuration:
  Drive Root    : /content/drive/MyDrive/EPGT_Research
  Runtime Root  : /content/EPGT_runtime
  Total Target  : 50,000 samples
  Platforms     : ['twitter', 'youtube', 'tiktok', 'instagram']


### Cell 2.2 — Membuat Struktur Folder Proyek

In [4]:
# ============================================================
# CELL 2.2 — Create Full Project Folder Structure
# ============================================================
# Membuat seluruh struktur folder secara otomatis.
# Operasi ini IDEMPOTENT — aman dijalankan berulang kali.

from pathlib import Path

# Daftar lengkap semua folder yang harus dibuat
FOLDERS = [
    # Data directories
    "data/raw/twitter",
    "data/raw/youtube",
    "data/raw/tiktok",
    "data/raw/instagram",
    "data/annotated/batch_01",
    "data/annotated/batch_02",
    "data/annotated/final",
    "data/processed",
    "data/features/text_embeddings",
    "data/features/emoji_features",
    "data/features/graph_objects",

    # Source code directories
    "src/data",
    "src/features",
    "src/models",
    "src/training",
    "src/evaluation",
    "src/utils",

    # Notebooks
    "notebooks",

    # Checkpoints
    "checkpoints/baselines/B1_indobert_textonly",
    "checkpoints/baselines/B2_bert_appended",
    "checkpoints/baselines/B3_cnn_classifier",
    "checkpoints/epgt/epoch_checkpoints",
    "checkpoints/ablation/ABL1_no_graph",
    "checkpoints/ablation/ABL2_no_fusion",
    "checkpoints/ablation/ABL3_no_emoji",
    "checkpoints/ablation/ABL4_no_position",
    "checkpoints/ablation/ABL5_full_epgt",

    # Logs
    "logs/wandb",
    "logs/training_logs",
    "logs/evaluation_reports",

    # Outputs
    "outputs/figures",
    "outputs/tables",
    "outputs/demo",
]

created = 0
existed = 0

for folder in FOLDERS:
    full_path = DRIVE_ROOT / folder
    if not full_path.exists():
        full_path.mkdir(parents=True, exist_ok=True)
        created += 1
    else:
        existed += 1

print("Project Folder Structure Created:")
print(f"  Folders created : {created}")
print(f"  Already existed : {existed}")
print(f"  Total folders   : {len(FOLDERS)}")
print(f"\nRoot: {DRIVE_ROOT}")

Project Folder Structure Created:
  Folders created : 0
  Already existed : 33
  Total folders   : 33

Root: /content/drive/MyDrive/EPGT_Research


Cell 2.3 — Membuat Template Source Code Files\:

In [5]:
# ============================================================
# CELL 2.3 — Create src/ Module Templates
# ============================================================
# Membuat file Python dengan template awal untuk setiap modul.
# File yang sudah ada TIDAK akan ditimpa.

from pathlib import Path

# ── Template definitions per file ─────────────────────────
SRC_FILES = {

"src/__init__.py": '''# EPGT Research Package
''',

"src/config.py": '''"""
config.py — Konfigurasi global proyek EPGT.
Semua path, hyperparameter, dan konstanta didefinisikan di sini.
Diimpor oleh semua modul lain sebagai single source of truth.
"""

from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict

# ── PATH CONFIGURATION ────────────────────────────────────
DRIVE_ROOT    = Path("/content/drive/MyDrive/EPGT_Research")
RUNTIME_ROOT  = Path("/content/EPGT_runtime")

DRIVE_PATHS = {
    "raw"         : DRIVE_ROOT / "data/raw",
    "annotated"   : DRIVE_ROOT / "data/annotated",
    "processed"   : DRIVE_ROOT / "data/processed",
    "features"    : DRIVE_ROOT / "data/features",
    "checkpoints" : DRIVE_ROOT / "checkpoints",
    "logs"        : DRIVE_ROOT / "logs",
    "outputs"     : DRIVE_ROOT / "outputs",
    "src"         : DRIVE_ROOT / "src",
}

RUNTIME_PATHS = {
    "data"        : RUNTIME_ROOT / "data",
    "processed"   : RUNTIME_ROOT / "data/processed",
    "features"    : RUNTIME_ROOT / "data/features",
    "checkpoints" : RUNTIME_ROOT / "checkpoints",
    "logs"        : RUNTIME_ROOT / "logs",
}

# ── DATASET CONFIGURATION ────────────────────────────────
@dataclass
class DatasetConfig:
    platform_targets: Dict[str, int] = field(default_factory=lambda: {
        "twitter": 20000, "youtube": 12500,
        "tiktok": 10000,  "instagram": 7500,
    })
    total_target          : int   = 50000
    collection_buffer     : float = 1.25
    train_ratio           : float = 0.70
    val_ratio             : float = 0.15
    test_ratio            : float = 0.15
    min_emoji_count       : int   = 1
    min_tokens_after_emoji: int   = 3
    max_text_length       : int   = 512
    min_iaa_kappa         : float = 0.70
    random_seed           : int   = 42

# ── MODEL CONFIGURATION ──────────────────────────────────
@dataclass
class ModelConfig:
    indobert_model_name  : str   = "indobenchmark/indobert-base-p1"
    text_embedding_dim   : int   = 768
    max_seq_length       : int   = 128
    emoji_embedding_dim  : int   = 200
    node_feature_dim     : int   = 203
    graph_hidden_dim     : int   = 256
    gat_heads            : int   = 4
    gat_layers           : int   = 2
    repetition_window_k  : int   = 3
    semantic_threshold   : float = 0.7
    combined_dim         : int   = 768
    dropout_rate         : float = 0.3
    num_intensity_classes: int   = 3
    num_sarcasm_classes  : int   = 2
    num_role_classes     : int   = 4

# ── TRAINING CONFIGURATION ───────────────────────────────
@dataclass
class TrainingConfig:
    optimizer             : str   = "adamw"
    learning_rate         : float = 2e-5
    batch_size            : int   = 16
    max_epochs            : int   = 20
    dropout               : float = 0.3
    weight_decay          : float = 0.01
    lambda_intensity      : float = 0.40
    lambda_sarcasm        : float = 0.35
    lambda_role           : float = 0.25
    early_stopping_patience  : int   = 3
    early_stopping_monitor   : str   = "val_macro_f1"
    early_stopping_min_delta : float = 0.001

# ── LABEL MAPPINGS ───────────────────────────────────────
INTENSITY_LABELS = {"Low": 0, "Medium": 1, "High": 2}
SARCASM_LABELS   = {"non_sarcastic": 0, "sarcastic": 1}
ROLE_LABELS      = {
    "Literal": 0, "Exaggeration": 1,
    "Irony": 2,   "Reaction": 3,
}

# Instantiate
dataset_config  = DatasetConfig()
model_config    = ModelConfig()
training_config = TrainingConfig()
''',

"src/data/__init__.py": '''# Data module
''',

"src/data/collector.py": '''"""
collector.py — Data collection dari 4 platform media sosial Indonesia.

Platform coverage:
  Twitter/X   : Tweepy + Twitter API v2
  YouTube     : Google API Python Client
  TikTok      : TikTok Research API / Playwright fallback
  Instagram   : Instaloader

Inclusion criteria (blueprint Section 2.1.3):
  - Bahasa Indonesia (incl. code-switching)
  - Minimum 1 emoji per post
  - Minimum 3 token setelah emoji dihapus
  - Bukan spam, bot, atau duplikat
"""

import re
import time
import hashlib
import logging
import pandas as pd
import emoji
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime, timezone

logger = logging.getLogger(__name__)


class InclusionFilter:
    """Filter sampel berdasarkan inclusion criteria blueprint."""

    def __init__(
        self,
        min_emoji_count: int = 1,
        min_token_count: int = 3,
    ):
        self.min_emoji_count = min_emoji_count
        self.min_token_count = min_token_count
        self._emoji_pattern  = re.compile(
            "(" + "|".join(
                re.escape(e)
                for e in sorted(emoji.EMOJI_DATA.keys(), key=len, reverse=True)
            ) + ")"
        )

    def passes(self, text: str) -> bool:
        if not text or not isinstance(text, str):
            return False
        emoji_count = len(self._emoji_pattern.findall(text))
        if emoji_count < self.min_emoji_count:
            return False
        cleaned    = self._emoji_pattern.sub("", text).strip()
        token_count = len(cleaned.split())
        if token_count < self.min_token_count:
            return False
        return True


class DuplicateFilter:
    """Deteksi dan hapus duplikat menggunakan hash teks."""

    def __init__(self):
        self._seen_hashes = set()

    def is_duplicate(self, text: str) -> bool:
        h = hashlib.md5(text.strip().lower().encode()).hexdigest()
        if h in self._seen_hashes:
            return True
        self._seen_hashes.add(h)
        return False

    def reset(self):
        self._seen_hashes.clear()


class TwitterCollector:
    """
    Collector untuk Twitter/X menggunakan Tweepy v4 + Twitter API v2.
    Target: 20,000 sampel (blueprint Section 2.1.1).
    """

    def __init__(self, bearer_token: str):
        import tweepy
        self.client = tweepy.Client(
            bearer_token=bearer_token,
            wait_on_rate_limit=True
        )
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def collect(
        self,
        queries       : List[str],
        target_count  : int = 20000,
        max_results   : int = 100,
    ) -> pd.DataFrame:
        """
        Kumpulkan tweet berbahasa Indonesia yang mengandung emoji.

        Args:
            queries      : List query Twitter API v2
            target_count : Jumlah sampel yang ingin dikumpulkan
            max_results  : Max results per request (10–100)

        Returns:
            DataFrame dengan kolom standar dataset
        """
        import tweepy
        records = []

        for query in queries:
            if len(records) >= target_count:
                break

            full_query = f"{query} lang:id -is:retweet -is:reply"
            logger.info(f"Twitter query: {full_query}")

            try:
                paginator = tweepy.Paginator(
                    self.client.search_recent_tweets,
                    query       = full_query,
                    tweet_fields= ["id", "text", "created_at", "lang"],
                    max_results = max_results,
                    limit       = (target_count // max_results) + 10
                )
                for response in paginator:
                    if not response.data:
                        continue
                    for tweet in response.data:
                        text = tweet.text
                        if self.dedup_filter.is_duplicate(text):
                            continue
                        if not self.inclusion_filter.passes(text):
                            continue
                        records.append({
                            "id"        : str(tweet.id),
                            "text"      : text,
                            "platform"  : "twitter",
                            "timestamp" : tweet.created_at.isoformat()
                                          if tweet.created_at else None,
                        })
                        if len(records) >= target_count:
                            break
                    time.sleep(0.5)

            except Exception as e:
                logger.error(f"Twitter collection error: {e}")
                continue

        logger.info(f"Twitter collected: {len(records)} samples")
        return pd.DataFrame(records)


class YouTubeCollector:
    """
    Collector untuk YouTube Comments menggunakan YouTube Data API v3.
    Target: 12,500 sampel (blueprint Section 2.1.1).
    Quota: 10,000 units/day. commentThreads.list = ~1 unit per request.
    """

    TARGET_KEYWORDS = [
        "gaming indonesia", "meme indonesia", "drama korea reaction",
        "review hp indonesia", "viral tiktok indonesia",
        "mukbang indonesia", "review makanan indonesia",
    ]

    def __init__(self, api_key: str):
        from googleapiclient.discovery import build
        self.youtube          = build("youtube", "v3", developerKey=api_key)
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def _search_video_ids(self, query: str, max_results: int = 50) -> List[str]:
        """Cari video IDs berdasarkan query."""
        try:
            response = self.youtube.search().list(
                q            = query,
                part         = "id",
                type         = "video",
                relevanceLanguage = "id",
                regionCode   = "ID",
                maxResults   = max_results,
            ).execute()
            return [
                item["id"]["videoId"]
                for item in response.get("items", [])
            ]
        except Exception as e:
            logger.error(f"YouTube search error: {e}")
            return []

    def _collect_comments(self, video_id: str, max_comments: int = 500) -> List[Dict]:
        """Ambil komentar dari satu video."""
        comments    = []
        page_token  = None

        while len(comments) < max_comments:
            try:
                kwargs = dict(
                    part       = "snippet",
                    videoId    = video_id,
                    maxResults = min(100, max_comments - len(comments)),
                    textFormat = "plainText",
                )
                if page_token:
                    kwargs["pageToken"] = page_token

                response   = self.youtube.commentThreads().list(**kwargs).execute()
                items      = response.get("items", [])

                for item in items:
                    snippet = item["snippet"]["topLevelComment"]["snippet"]
                    text    = snippet.get("textOriginal", "")
                    if not text or self.dedup_filter.is_duplicate(text):
                        continue
                    if not self.inclusion_filter.passes(text):
                        continue
                    comments.append({
                        "id"        : item["id"],
                        "text"      : text,
                        "platform"  : "youtube",
                        "timestamp" : snippet.get("publishedAt"),
                    })

                page_token = response.get("nextPageToken")
                if not page_token:
                    break
                time.sleep(0.3)

            except Exception as e:
                logger.error(f"YouTube comment error (video={video_id}): {e}")
                break

        return comments

    def collect(
        self,
        target_count: int = 12500,
    ) -> pd.DataFrame:
        records = []
        for keyword in self.TARGET_KEYWORDS:
            if len(records) >= target_count:
                break
            video_ids = self._search_video_ids(keyword, max_results=20)
            logger.info(f"YouTube keyword \'{keyword}\': {len(video_ids)} videos")
            for vid_id in video_ids:
                if len(records) >= target_count:
                    break
                comments = self._collect_comments(vid_id, max_comments=300)
                records.extend(comments)
                time.sleep(1.0)

        logger.info(f"YouTube collected: {len(records)} samples")
        return pd.DataFrame(records[:target_count])


class TikTokCollector:
    """
    Collector untuk TikTok Comments.
    Menggunakan Playwright untuk scraping komentar publik.
    Rate limiting ketat: 1 request per 3-5 detik.
    """

    TARGET_HASHTAGS = [
        "fyp", "fypシ", "viral", "trending",
        "meme", "indonesia", "receh",
    ]

    def __init__(self):
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def collect(
        self,
        target_count: int = 10000,
        use_mock    : bool = False,
    ) -> pd.DataFrame:
        """
        Kumpulkan komentar TikTok.
        Set use_mock=True untuk generate data simulasi saat API tidak tersedia.
        """
        if use_mock:
            return self._generate_mock_data(target_count)
        return self._collect_via_playwright(target_count)

    def _collect_via_playwright(self, target_count: int) -> pd.DataFrame:
        """Scraping TikTok via Playwright browser automation."""
        import asyncio
        from playwright.async_api import async_playwright

        records = []
        # Implementation placeholder — diisi sesuai TikTok URL structure
        logger.warning(
            "TikTok Playwright collector requires manual session setup. "
            "Use use_mock=True for development/testing."
        )
        return pd.DataFrame(records)

    def _generate_mock_data(self, count: int) -> pd.DataFrame:
        """Generate data simulasi untuk development dan testing pipeline."""
        import random
        MOCK_TEMPLATES = [
            "mantap banget sih ini 🗿🗿 gampang katanya",
            "wkwk lucu parah 😂😂😂 receh abis",
            "sedih banget deh 😭 kok bisa gitu",
            "keren sih emang 🔥🔥 respect banget",
            "gak nyangka beneran viral 💀 wtf",
            "oke deh 👍 lanjut aja gitu",
            "haha ngakak 😹 receh banget sumpah",
            "sebenernya bagus sih 🤔 tapi ya gitu deh",
            "yasalam beneran gak nyambung 😑 awkward",
            "literally best day ever 🥰✨ happy banget",
        ]
        random.seed(42)
        records = []
        for i in range(count):
            records.append({
                "id"       : f"tiktok_mock_{i:06d}",
                "text"     : random.choice(MOCK_TEMPLATES),
                "platform" : "tiktok",
                "timestamp": datetime.now(timezone.utc).isoformat(),
            })
        logger.info(f"TikTok mock generated: {len(records)} samples")
        return pd.DataFrame(records)


class InstagramCollector:
    """
    Collector untuk Instagram Comments menggunakan Instaloader.
    Target: 7,500 sampel (blueprint Section 2.1.1).
    """

    TARGET_ACCOUNTS = [
        "awkarin", "rahmawatikeisyaofficial", "awkarin",
        "felicyangelista_", "awkarin",
    ]

    def __init__(
        self,
        username: Optional[str] = None,
        password: Optional[str] = None,
    ):
        import instaloader
        self.loader           = instaloader.Instaloader()
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

        if username and password:
            try:
                self.loader.login(username, password)
                logger.info(f"Instagram logged in as {username}")
            except Exception as e:
                logger.warning(f"Instagram login failed: {e}. Using anonymous.")

    def collect(
        self,
        target_accounts : List[str] = None,
        target_count    : int = 7500,
        use_mock        : bool = False,
    ) -> pd.DataFrame:
        if use_mock:
            return self._generate_mock_data(target_count)

        import instaloader
        accounts = target_accounts or self.TARGET_ACCOUNTS
        records  = []

        for account in accounts:
            if len(records) >= target_count:
                break
            try:
                profile = instaloader.Profile.from_username(
                    self.loader.context, account
                )
                for post in profile.get_posts():
                    if len(records) >= target_count:
                        break
                    for comment in post.get_comments():
                        text = comment.text
                        if self.dedup_filter.is_duplicate(text):
                            continue
                        if not self.inclusion_filter.passes(text):
                            continue
                        records.append({
                            "id"       : str(comment.id),
                            "text"     : text,
                            "platform" : "instagram",
                            "timestamp": comment.created_at_utc.isoformat(),
                        })
                    time.sleep(2.0)   # Instaloader rate limit
            except Exception as e:
                logger.error(f"Instagram error ({account}): {e}")
                continue

        logger.info(f"Instagram collected: {len(records)} samples")
        return pd.DataFrame(records[:target_count])

    def _generate_mock_data(self, count: int) -> pd.DataFrame:
        import random
        MOCK_TEMPLATES = [
            "cantik banget 😍😍 goals bgt sih",
            "outfitnya kece 🔥 mau dong ditag brandnya",
            "ih lucu banget 🥰✨ gemoy",
            "ini tempat makan apa? 🤤 mau kesana",
            "serius ini enak 😭 pengen nyoba",
        ]
        random.seed(42)
        records = [
            {
                "id"       : f"ig_mock_{i:06d}",
                "text"     : random.choice(MOCK_TEMPLATES),
                "platform" : "instagram",
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            for i in range(count)
        ]
        return pd.DataFrame(records)
''',

"src/data/preprocessor.py": '''"""
preprocessor.py — Pipeline preprocessing teks Indonesia untuk EPGT.
Implementasi lengkap di NB03. File ini adalah placeholder.
"""
# TODO: Implemented in NB03
''',

"src/features/__init__.py": '''# Features module
''',

"src/features/emoji_graph.py": '''"""
emoji_graph.py — Konstruksi Emoji Pragmatic Graph G=(V,E,W).
Implementasi lengkap di NB03. File ini adalah placeholder.
"""
# TODO: Implemented in NB03
''',

"src/models/__init__.py": '''# Models module
''',

"src/models/text_encoder.py": '''"""
text_encoder.py — IndoBERT Text Semantic Encoder.
Component 1 dari arsitektur EPGT.
Output: text_embedding ∈ R^768 (CLS token pooled).
Implementasi lengkap di NB04.
"""
# TODO: Implemented in NB04
''',

"src/models/gat_encoder.py": '''"""
gat_encoder.py — 2-layer Graph Attention Network Encoder.
Component 2 dari arsitektur EPGT.
Forward pass dengan edge weight modulation sesuai blueprint.
Output: graph_embedding ∈ R^256 (MeanPool over nodes).
Implementasi lengkap di NB04.
"""
# TODO: Implemented in NB04
''',

"src/models/fusion_layer.py": '''"""
fusion_layer.py — Pragmatic Fusion Layer (Cross-Attention).
Component 3 dari arsitektur EPGT.
Q=text, K/V=emoji_graph → combined_repr ∈ R^768.
Implementasi lengkap di NB04.
"""
# TODO: Implemented in NB04
''',

"src/models/classification_head.py": '''"""
classification_head.py — Multi-Task Classification Head.
Component 4 dari arsitektur EPGT.
Head A: 3-class softmax (intensity)
Head B: Binary sigmoid (sarcasm)
Head C: 4-class softmax (emoji role)
Implementasi lengkap di NB04.
"""
# TODO: Implemented in NB04
''',

"src/models/epgt.py": '''"""
epgt.py — Full EPGT Model Assembly.
Mengintegrasikan keempat komponen arsitektur:
  Component 1: TextSemanticEncoder (IndoBERT)
  Component 2: EmojiGraphEncoder (2-layer GAT)
  Component 3: PragmaticFusionLayer (Cross-Attention)
  Component 4: MTLClassificationHead (3 parallel heads)
Mendukung ablation_mode: no_graph, no_fusion, no_emoji, no_position.
Implementasi lengkap di NB04.
"""
# TODO: Implemented in NB04
''',

"src/training/__init__.py": '''# Training module
''',

"src/training/loss.py": '''"""
loss.py — Weighted Multi-Task Learning Loss.
L_total = 0.4*L_intensity + 0.35*L_sarcasm + 0.25*L_role
Lambda coefficients sebagai tunable hyperparameter.
Implementasi lengkap di NB05.
"""
# TODO: Implemented in NB05
''',

"src/training/trainer.py": '''"""
trainer.py — MTL Training Loop dengan Early Stopping.
Monitor: val_macro_f1. Patience: 3 epochs.
Checkpoint management via DriveManager.
W&B logging per step dan per epoch.
Implementasi lengkap di NB05.
"""
# TODO: Implemented in NB05
''',

"src/evaluation/__init__.py": '''# Evaluation module
''',

"src/evaluation/metrics.py": '''"""
metrics.py — Evaluation metrics untuk semua eksperimen EPGT.
General: Accuracy, Precision, Recall, Macro F1
Sarcasm: ROC-AUC, PR-AUC, MCC (extended metrics)
Ablation: Delta F1 per component
Implementasi lengkap di NB06.
"""
# TODO: Implemented in NB06
''',

"src/utils/__init__.py": '''# Utils module
''',

"src/utils/drive_manager.py": '''"""
drive_manager.py — Google Drive <-> Colab Runtime Sync Manager.

Arsitektur dua-tier:
  TIER 1 (Persistent) : Google Drive  - penyimpanan jangka panjang
  TIER 2 (Fast)       : /content/     - I/O cepat saat training

Key methods:
  mount_drive()              - Mount Drive ke /content/drive
  setup_project_structure()  - Buat folder struktur proyek
  sync_to_runtime()          - Copy data Drive -> Runtime (pre-training)
  sync_checkpoint_to_drive() - Copy checkpoint Runtime -> Drive (post-epoch)
  sync_logs_to_drive()       - Copy logs Runtime -> Drive
  verify_sync()              - Verifikasi file kritis tersedia di runtime
"""

import os
import shutil
import time
from pathlib import Path
from typing import List, Optional, Dict


class DriveManager:
    """Mengelola sinkronisasi antara Google Drive dan Colab runtime."""

    def __init__(
        self,
        drive_root  : str = "/content/drive/MyDrive/EPGT_Research",
        runtime_root: str = "/content/EPGT_runtime",
    ):
        self.drive_root   = Path(drive_root)
        self.runtime_root = Path(runtime_root)
        self._is_mounted  = False

    def mount_drive(self, force_remount: bool = False) -> None:
        """Mount Google Drive."""
        from google.colab import drive
        mount_path = "/content/drive"
        if os.path.exists(mount_path) and not force_remount:
            print("Google Drive sudah ter-mount.")
        else:
            drive.mount(mount_path, force_remount=force_remount)
        self._is_mounted = True

    def sync_to_runtime(
        self,
        sync_targets   : Optional[List[str]] = None,
        show_progress  : bool = True,
    ) -> Dict[str, float]:
        """Salin dataset dari Drive ke runtime lokal untuk I/O cepat."""
        if sync_targets is None:
            sync_targets = ["data/processed", "data/features"]

        self.runtime_root.mkdir(parents=True, exist_ok=True)
        stats   = {}
        t_start = time.time()

        print("\nSyncing: Google Drive -> Colab Runtime")
        for target in sync_targets:
            src = self.drive_root / target
            dst = self.runtime_root / target
            if not src.exists():
                print(f"  SKIP {target} (not found)")
                continue
            t0  = time.time()
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(str(src), str(dst))
            size_mb      = self._dir_size_mb(dst)
            stats[target]= size_mb
            if show_progress:
                print(f"  {target}: {size_mb:.1f} MB in {time.time()-t0:.1f}s")

        total_mb = sum(stats.values())
        print(f"Total: {total_mb:.1f} MB in {time.time()-t_start:.1f}s")
        return stats

    def sync_checkpoint_to_drive(
        self, checkpoint_name: str, local_path: str
    ) -> None:
        """Simpan checkpoint dari runtime ke Drive."""
        dst_dir = self.drive_root / "checkpoints" / checkpoint_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        src = Path(local_path)
        shutil.copy2(str(src), str(dst_dir / src.name))
        print(f"Checkpoint saved: {dst_dir / src.name}")

    def sync_logs_to_drive(self, local_log_dir: str) -> None:
        """Sinkronisasi log dari runtime ke Drive."""
        src = Path(local_log_dir)
        if not src.exists():
            return
        dst = self.drive_root / "logs" / "training_logs"
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.glob("*.csv"):
            shutil.copy2(str(f), str(dst / f.name))
        print(f"Logs synced to Drive: {dst}")

    def verify_sync(self) -> Dict[str, bool]:
        """Verifikasi file kritis tersedia di runtime sebelum training."""
        critical = [
            "data/processed/train.csv",
            "data/processed/val.csv",
            "data/processed/test.csv",
        ]
        results = {}
        print("Verifying runtime data:")
        for f in critical:
            exists      = (self.runtime_root / f).exists()
            results[f]  = exists
            print(f"  [{\"OK\" if exists else \"MISSING\"}] {f}")
        return results

    def get_runtime_path(self, relative_path: str) -> Path:
        return self.runtime_root / relative_path

    def get_drive_path(self, relative_path: str) -> Path:
        return self.drive_root / relative_path

    def _dir_size_mb(self, path: Path) -> float:
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
''',

"src/utils/seed.py": '''"""
seed.py — Reproducibility seed control.
"""
import os, random
import numpy as np

def set_seed(seed: int = 42) -> None:
    """Set semua random seeds untuk reprodusibilitas eksperimen."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    except ImportError:
        pass
    print(f"Random seed set: {seed}")
''',

"src/utils/logger.py": '''"""
logger.py — Weights & Biases logging wrapper untuk EPGT.
Implementasi lengkap di NB05.
"""
# TODO: Implemented in NB05
''',

}

# ── Tulis semua file ───────────────────────────────────────
created  = 0
skipped  = 0

for rel_path, content in SRC_FILES.items():
    full_path = DRIVE_ROOT / rel_path
    full_path.parent.mkdir(parents=True, exist_ok=True)

    if not full_path.exists():
        full_path.write_text(content, encoding="utf-8")
        created += 1
        print(f"  [CREATED] {rel_path}")
    else:
        skipped += 1
        print(f"  [EXISTS]  {rel_path}")

print(f"\nSource files created : {created}")
print(f"Source files skipped : {skipped} (already exist)")

  [EXISTS]  src/__init__.py
  [EXISTS]  src/config.py
  [EXISTS]  src/data/__init__.py
  [EXISTS]  src/data/collector.py
  [EXISTS]  src/data/preprocessor.py
  [EXISTS]  src/features/__init__.py
  [EXISTS]  src/features/emoji_graph.py
  [EXISTS]  src/models/__init__.py
  [EXISTS]  src/models/text_encoder.py
  [EXISTS]  src/models/gat_encoder.py
  [EXISTS]  src/models/fusion_layer.py
  [EXISTS]  src/models/classification_head.py
  [EXISTS]  src/models/epgt.py
  [EXISTS]  src/training/__init__.py
  [EXISTS]  src/training/loss.py
  [EXISTS]  src/training/trainer.py
  [EXISTS]  src/evaluation/__init__.py
  [EXISTS]  src/evaluation/metrics.py
  [EXISTS]  src/utils/__init__.py
  [EXISTS]  src/utils/drive_manager.py
  [EXISTS]  src/utils/seed.py
  [EXISTS]  src/utils/logger.py

Source files created : 0
Source files skipped : 22 (already exist)


### Cell 2.4 — Verifikasi Struktur Proyek

In [6]:
# ============================================================
# CELL 2.4 — Verify Project Structure
# ============================================================

from pathlib import Path

def print_tree(path: Path, prefix: str = "", max_depth: int = 3, depth: int = 0):
    """Print struktur folder seperti perintah `tree`."""
    if depth > max_depth:
        return
    items = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name))
    for i, item in enumerate(items):
        connector = "└── " if i == len(items) - 1 else "├── "
        print(f"{prefix}{connector}{item.name}")
        if item.is_dir() and depth < max_depth:
            extension = "    " if i == len(items) - 1 else "│   "
            print_tree(item, prefix + extension, max_depth, depth + 1)

print(f"EPGT_Research/")
print_tree(DRIVE_ROOT, max_depth=3)

EPGT_Research/
├── checkpoints
│   ├── ablation
│   │   ├── ABL1_no_graph
│   │   ├── ABL2_no_fusion
│   │   ├── ABL3_no_emoji
│   │   ├── ABL4_no_position
│   │   └── ABL5_full_epgt
│   ├── baselines
│   │   ├── B1_indobert_textonly
│   │   ├── B2_bert_appended
│   │   └── B3_cnn_classifier
│   └── epgt
│       └── epoch_checkpoints
├── data
│   ├── annotated
│   │   ├── batch_01
│   │   ├── batch_02
│   │   └── final
│   ├── features
│   │   ├── emoji_features
│   │   ├── graph_objects
│   │   └── text_embeddings
│   ├── processed
│   └── raw
│       ├── instagram
│       │   └── raw_instagram.csv
│       ├── tiktok
│       │   └── raw_tiktok.csv
│       ├── twitter
│       │   └── raw_twitter.csv
│       └── youtube
│           └── raw_youtube.csv
├── logs
│   ├── evaluation_reports
│   ├── training_logs
│   └── wandb
├── notebooks
├── outputs
│   ├── demo
│   ├── figures
│   └── tables
└── src
    ├── __pycache__
    │   └── config.cpython-312.pyc
    ├── data
    │   ├── __pycache

### Cell 2.5 — Inject src/ ke Python Path

In [7]:
# ============================================================
# CELL 2.5 — Setup Python Path
# ============================================================
# Wajib dijalankan di SEMUA notebook agar import dari src/ berfungsi.

import sys
from pathlib import Path

SRC_PATH = str(DRIVE_ROOT / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
    print(f"Added to sys.path: {SRC_PATH}")
else:
    print(f"Already in sys.path: {SRC_PATH}")

# Verifikasi import
try:
    from config import dataset_config, model_config, training_config
    print("\nImport test PASSED:")
    print(f"  Dataset target  : {dataset_config.total_target:,} samples")
    print(f"  IndoBERT model  : {model_config.indobert_model_name}")
    print(f"  Learning rate   : {training_config.learning_rate}")
except Exception as e:
    print(f"Import test FAILED: {e}")

Added to sys.path: /content/drive/MyDrive/EPGT_Research/src

Import test PASSED:
  Dataset target  : 50,000 samples
  IndoBERT model  : indobenchmark/indobert-base-p1
  Learning rate   : 2e-05


---
## BAGIAN 3 — API CREDENTIALS SETUP
### Cell 3.1 — Konfigurasi API Keys

In [8]:
# ============================================================
# CELL 3.1 — API Credentials Configuration
# ============================================================
# INSTRUKSI:
#   Masukkan API keys Anda di bawah ini.
#   JANGAN commit file ini ke version control dengan key yang terisi.
#   Gunakan Colab Secrets (Tools > Secrets) untuk production.
#
# Cara menggunakan Colab Secrets:
#   from google.colab import userdata
#   TWITTER_BEARER = userdata.get('TWITTER_BEARER_TOKEN')

from google.colab import userdata

# ── Cara 1: Via Colab Secrets (RECOMMENDED) ───────────────
try:
    TWITTER_BEARER_TOKEN = userdata.get("TWITTER_BEARER_TOKEN")
    YOUTUBE_API_KEY      = userdata.get("YOUTUBE_API_KEY")
    INSTAGRAM_USERNAME   = userdata.get("INSTAGRAM_USERNAME")
    INSTAGRAM_PASSWORD   = userdata.get("INSTAGRAM_PASSWORD")
    print("Credentials loaded from Colab Secrets.")
except Exception:
    # ── Cara 2: Manual Input (fallback) ───────────────────
    print("Colab Secrets tidak tersedia. Menggunakan manual input.")
    print("Kosongkan jika platform tidak digunakan (akan pakai mock data).")
    TWITTER_BEARER_TOKEN = ""   # Isi dengan bearer token Twitter
    YOUTUBE_API_KEY      = ""   # Isi dengan YouTube Data API v3 key
    INSTAGRAM_USERNAME   = ""   # Isi dengan username Instagram
    INSTAGRAM_PASSWORD   = ""   # Isi dengan password Instagram

# ── Status report ─────────────────────────────────────────
print("\nAPI Credentials Status:")
print(f"  Twitter Bearer Token : {'SET' if TWITTER_BEARER_TOKEN else 'NOT SET (will use mock)'}")
print(f"  YouTube API Key      : {'SET' if YOUTUBE_API_KEY else 'NOT SET (will use mock)'}")
print(f"  Instagram Username   : {'SET' if INSTAGRAM_USERNAME else 'NOT SET (will use mock)'}")

Colab Secrets tidak tersedia. Menggunakan manual input.
Kosongkan jika platform tidak digunakan (akan pakai mock data).

API Credentials Status:
  Twitter Bearer Token : NOT SET (will use mock)
  YouTube API Key      : NOT SET (will use mock)
  Instagram Username   : NOT SET (will use mock)


###CELL 3.5 — FIX: Patch collector.py

In [9]:
# ============================================================
# CELL 3.5 — FIX: Patch collector.py
# Tambahkan cell baru SETELAH Cell 3.1 (API Credentials)
# Jalankan SATU KALI sebelum Cell 4.1
# ============================================================

from pathlib import Path

DRIVE_ROOT     = Path("/content/drive/MyDrive/EPGT_Research")
collector_path = DRIVE_ROOT / "src/data/collector.py"

COLLECTOR_CONTENT = '''\
"""
collector.py — Data collection dari 4 platform media sosial Indonesia.

Platform coverage:
  Twitter/X   : Tweepy + Twitter API v2          | ID prefix: twitter_
  YouTube     : Google API Python Client          | ID prefix: youtube_
  TikTok      : Research API / Playwright         | ID prefix: tiktok_
  Instagram   : Instaloader                       | ID prefix: ig_

Mock data design (saat API tidak tersedia):
  - Setiap platform punya ID prefix unik
  - Setiap platform punya 20 template teks yang berbeda
  - Setiap platform punya random seed yang berbeda (11/22/33/44)
  - generate_mock() adalah method per class, bukan shared function
"""

import re
import time
import hashlib
import logging
import pandas as pd
import emoji
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime, timezone

logger = logging.getLogger(__name__)


TWITTER_TEMPLATES = [
    "mantap banget sih ini 🗿🗿 gampang katanya",
    "wkwk lucu parah 😂😂😂 receh abis deh",
    "sedih banget deh 😭 kok bisa gitu sih",
    "keren sih emang 🔥🔥 respect banget lah",
    "gak nyangka beneran viral 💀 wtf banget",
    "ya ampun ini mah beneran parah 😤 kesel",
    "serius gak sih 🤔 masa iya gitu caranya",
    "gaskeun lah 💪🔥 siap jalan terus",
    "mager banget hari ini 😪 gabut poll",
    "astaga beneran gak nyambung 😑 awkward",
    "literally best thing ever 🥰✨ seneng banget",
    "oke deh terserah 👍 udah gitu aja",
    "kepo banget sama ini 👀 penasaran banget",
    "ngakak parah 😹😹 gak ketahan ketawa",
    "healing dulu yuk 🌿😌 butuh refreshing",
    "bucin parah emang 💔 gapapa sih",
    "receh bet ini kontennya 😂 tapi lucu juga",
    "gercep dong jangan lama 😤 keburu basi",
    "ya elah masa gitu 🙄 gak masuk akal",
    "mantap jiwa 🫡 respect tinggi buat ini",
]

YOUTUBE_TEMPLATES = [
    "konten ini bagus banget sih 👍👍 subscribe dulu ah",
    "wkwk ngakak parah nonton ini 😂😂 lucu bet",
    "keren banget editannya 🔥✨ respect sama creator",
    "relate banget sama situasi ini 😭😭 real talk",
    "gilak beneran gak nyangka plot twistnya 💀 anjir",
    "tutorial ini helpful banget 🙏✨ makasih banyak",
    "gak nyangka bisa sebagus ini 😍 submas dulu ah",
    "receh tapi menghibur 😄 terus berkarya ya kak",
    "serius bagus banget content quality nya 🎬🔥 top",
    "ini mah beneran bikin nangis 😢💔 sedih parah",
    "sumpah lucu banget 😂🤣 nonton berkali kali",
    "mantap bet ini video 💯🔥 gas terus kontennya",
    "akhirnya ada yang bahas ini 👏👏 nunggu lama",
    "beneran detail banget penjelasannya 📚✨ keren",
    "gak bosen nonton ini 🎵😍 enak banget emang",
    "ya ampun baru tau ada yang kayak gini 😲👀",
    "edit video ini smooth banget 🎬✨ aesthetic parah",
    "haha parah ini 😂😂😂 lawak banget sih",
    "informasinya bermanfaat banget 🙏📖 makasih kak",
    "pengen coba juga nih 🤩💪 inspiratif banget",
]

TIKTOK_TEMPLATES = [
    "mantap banget sih 🗿 gampang banget katanya",
    "wkwkwk 💀💀 ga ada obatnya emang",
    "fyp dong please 🙏🙏 udah ngebantu spread",
    "ini aku banget 😭✋ relate parah sumpah",
    "gak ada yang nanya tapi aku jawab 🗿",
    "beneran gak ketawa 💀 serius ini",
    "skill issue 🗿 maaf ya",
    "ngapain sih 😭😭 ga ada kerjaan",
    "literally me everyday 💀😭",
    "ratio plus L plus skill issue 🗿",
    "oke sip mantap 🫡 lanjut",
    "ga ada yang minta pendapat lo 🗿",
    "aduh kenapa gini sih 😭🙏 capek",
    "ini real atau editan 👀👀 penasaran",
    "kenapaaaa 😭💀 aku mau nangis",
    "itu aku yang komentar 🫵😂 ketahuan",
    "lanjutkan kak 🔥🔥 bikin konten terus",
    "aku gak ngerti 🤡 mungkin aku yang bego",
    "plot twist terkejut 😲💀 ga nyangka",
    "receh tapi bener juga sih 🗿 mikir",
]

INSTAGRAM_TEMPLATES = [
    "cantik banget 😍😍 goals bgt sih aesthetic",
    "outfit kece banget 🔥 mau dong info brandnya",
    "ih lucu banget 🥰✨ gemoy sumpah",
    "ini tempat makan dimana? 🤤 pengen kesana",
    "serius ini enak banget 😭😋 pengen nyoba",
    "foto bagus banget editannya 📸✨ preset apa ini?",
    "glow up parah 😍🌟 cantik banget sekarang",
    "tempat ini aesthetic banget 🌸📍 mau kesini",
    "ootd keren banget 👗🔥 inspirasi banget sih",
    "makanan ini enak banget kayaknya 🍜😍 wajib coba",
    "view nya indah banget 🌅✨ healing spot banget",
    "beneran bagus banget hasilnya 💄😍 skincare apa?",
    "caption ini relate banget 💕🥺 touching banget",
    "ini beneran natural? 😲✨ glowing banget",
    "koleksinya lucu lucu 🛍️😍 mau semua",
    "feed nya aesthetic banget 🎨✨ konsisten",
    "resepnya boleh dong share 🍳😋 keliatan enak",
    "traveling goals banget 🌏✈️ pengen kesini",
    "couple goals banget 💑❤️ sweet parah",
    "hair goals banget 💇✨ salon mana ini?",
]


def _build_dataframe(records, platform, id_prefix):
    for i, rec in enumerate(records):
        rec["id"]       = f"{id_prefix}_{i:06d}"
        rec["platform"] = platform
    return pd.DataFrame(records)


class InclusionFilter:
    def __init__(self, min_emoji_count=1, min_token_count=3):
        self.min_emoji_count = min_emoji_count
        self.min_token_count = min_token_count
        self._pat = re.compile(
            "(" + "|".join(
                re.escape(e)
                for e in sorted(emoji.EMOJI_DATA.keys(), key=len, reverse=True)
            ) + ")"
        )

    def passes(self, text):
        if not text or not isinstance(text, str):
            return False
        if len(self._pat.findall(text)) < self.min_emoji_count:
            return False
        cleaned = self._pat.sub("", text).strip()
        return len(cleaned.split()) >= self.min_token_count


class DuplicateFilter:
    def __init__(self):
        self._seen = set()

    def is_duplicate(self, text):
        h = hashlib.md5(text.strip().lower().encode()).hexdigest()
        if h in self._seen:
            return True
        self._seen.add(h)
        return False

    def reset(self):
        self._seen.clear()


class TwitterCollector:
    def __init__(self, bearer_token):
        import tweepy
        self.client           = tweepy.Client(bearer_token=bearer_token, wait_on_rate_limit=True)
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def collect(self, queries, target_count=20000, max_results=100):
        import tweepy
        records = []
        for query in queries:
            if len(records) >= target_count:
                break
            full_query = f"{query} lang:id -is:retweet -is:reply"
            try:
                paginator = tweepy.Paginator(
                    self.client.search_recent_tweets,
                    query=full_query,
                    tweet_fields=["id", "text", "created_at"],
                    max_results=max_results,
                    limit=(target_count // max_results) + 10,
                )
                for response in paginator:
                    if not response.data:
                        continue
                    for tweet in response.data:
                        text = tweet.text
                        if self.dedup_filter.is_duplicate(text):
                            continue
                        if not self.inclusion_filter.passes(text):
                            continue
                        records.append({
                            "text"     : text,
                            "timestamp": tweet.created_at.isoformat() if tweet.created_at else None,
                        })
                        if len(records) >= target_count:
                            break
                    time.sleep(0.5)
            except Exception as e:
                logger.error(f"Twitter error: {e}")
                continue
        return _build_dataframe(records, "twitter", "twitter")

    def generate_mock(self, count):
        import random
        random.seed(11)
        records = [
            {"text": random.choice(TWITTER_TEMPLATES), "timestamp": datetime.now(timezone.utc).isoformat()}
            for _ in range(count)
        ]
        return _build_dataframe(records, "twitter", "twitter")


class YouTubeCollector:
    TARGET_KEYWORDS = [
        "gaming indonesia", "meme indonesia", "drama korea reaction",
        "review hp indonesia", "viral tiktok indonesia",
        "mukbang indonesia", "review makanan indonesia",
    ]

    def __init__(self, api_key):
        from googleapiclient.discovery import build
        self.youtube          = build("youtube", "v3", developerKey=api_key)
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def collect(self, target_count=12500):
        records = []
        for keyword in self.TARGET_KEYWORDS:
            if len(records) >= target_count:
                break
            for vid_id in self._search_video_ids(keyword):
                if len(records) >= target_count:
                    break
                records.extend(self._collect_comments(vid_id))
                time.sleep(1.0)
        return _build_dataframe(records[:target_count], "youtube", "youtube")

    def _search_video_ids(self, query, max_results=20):
        try:
            response = self.youtube.search().list(
                q=query, part="id", type="video",
                relevanceLanguage="id", regionCode="ID",
                maxResults=max_results,
            ).execute()
            return [item["id"]["videoId"] for item in response.get("items", [])]
        except Exception as e:
            logger.error(f"YouTube search error: {e}")
            return []

    def _collect_comments(self, video_id, max_comments=300):
        comments, page_token = [], None
        while len(comments) < max_comments:
            try:
                kwargs = dict(part="snippet", videoId=video_id,
                              maxResults=min(100, max_comments - len(comments)),
                              textFormat="plainText")
                if page_token:
                    kwargs["pageToken"] = page_token
                response = self.youtube.commentThreads().list(**kwargs).execute()
                for item in response.get("items", []):
                    snippet = item["snippet"]["topLevelComment"]["snippet"]
                    text    = snippet.get("textOriginal", "")
                    if not text or self.dedup_filter.is_duplicate(text):
                        continue
                    if not self.inclusion_filter.passes(text):
                        continue
                    comments.append({"text": text, "timestamp": snippet.get("publishedAt")})
                page_token = response.get("nextPageToken")
                if not page_token:
                    break
                time.sleep(0.3)
            except Exception as e:
                logger.error(f"YouTube comment error: {e}")
                break
        return comments

    def generate_mock(self, count):
        import random
        random.seed(22)
        records = [
            {"text": random.choice(YOUTUBE_TEMPLATES), "timestamp": datetime.now(timezone.utc).isoformat()}
            for _ in range(count)
        ]
        return _build_dataframe(records, "youtube", "youtube")


class TikTokCollector:
    def __init__(self):
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()

    def collect(self, target_count=10000, use_mock=False):
        if use_mock:
            return self.generate_mock(target_count)
        logger.warning("TikTok Playwright requires manual session. Use use_mock=True.")
        return pd.DataFrame()

    def generate_mock(self, count):
        import random
        random.seed(33)
        records = [
            {"text": random.choice(TIKTOK_TEMPLATES), "timestamp": datetime.now(timezone.utc).isoformat()}
            for _ in range(count)
        ]
        return _build_dataframe(records, "tiktok", "tiktok")


class InstagramCollector:
    def __init__(self, username=None, password=None):
        self.inclusion_filter = InclusionFilter()
        self.dedup_filter     = DuplicateFilter()
        self._username        = username
        self._password        = password

    def collect(self, target_accounts=None, target_count=7500, use_mock=False):
        if use_mock or not (self._username and self._password):
            return self.generate_mock(target_count)
        import instaloader
        loader, records = instaloader.Instaloader(), []
        try:
            loader.login(self._username, self._password)
        except Exception as e:
            logger.warning(f"Instagram login failed: {e}. Using mock.")
            return self.generate_mock(target_count)
        for account in (target_accounts or []):
            if len(records) >= target_count:
                break
            try:
                profile = instaloader.Profile.from_username(loader.context, account)
                for post in profile.get_posts():
                    if len(records) >= target_count:
                        break
                    for comment in post.get_comments():
                        text = comment.text
                        if self.dedup_filter.is_duplicate(text):
                            continue
                        if not self.inclusion_filter.passes(text):
                            continue
                        records.append({"text": text, "timestamp": comment.created_at_utc.isoformat()})
                    time.sleep(2.0)
            except Exception as e:
                logger.error(f"Instagram error ({account}): {e}")
        if not records:
            return self.generate_mock(target_count)
        return _build_dataframe(records[:target_count], "instagram", "ig")

    def generate_mock(self, count):
        import random
        random.seed(44)
        records = [
            {"text": random.choice(INSTAGRAM_TEMPLATES), "timestamp": datetime.now(timezone.utc).isoformat()}
            for _ in range(count)
        ]
        return _build_dataframe(records, "instagram", "ig")
'''

collector_path.parent.mkdir(parents=True, exist_ok=True)
collector_path.write_text(COLLECTOR_CONTENT, encoding="utf-8")

print("collector.py patched successfully.")
print(f"Path : {collector_path}")
print(f"Size : {collector_path.stat().st_size:,} bytes")
print()
print("Perubahan yang diterapkan:")
print("  ID prefix unik  : twitter_ / youtube_ / tiktok_ / ig_")
print("  Random seed     : Twitter=11, YouTube=22, TikTok=33, Instagram=44")
print("  Template teks   : 20 template berbeda per platform")
print()
print("Lanjutkan ke Cell 4.1.")

collector.py patched successfully.
Path : /content/drive/MyDrive/EPGT_Research/src/data/collector.py
Size : 14,363 bytes

Perubahan yang diterapkan:
  ID prefix unik  : twitter_ / youtube_ / tiktok_ / ig_
  Random seed     : Twitter=11, YouTube=22, TikTok=33, Instagram=44
  Template teks   : 20 template berbeda per platform

Lanjutkan ke Cell 4.1.


---
## BAGIAN 4 — DATA COLLECTION PIPELINE
### Cell 4.1 — Twitter/X Data Collection

In [10]:
# ============================================================
# CELL 4.1 — Twitter/X Data Collection
# Target: 20,000 samples (40% of total dataset)
# ============================================================

import sys, logging
from pathlib import Path
from importlib import reload

DRIVE_ROOT = Path("/content/drive/MyDrive/EPGT_Research")
sys.path.insert(0, str(DRIVE_ROOT / "src"))

import data.collector as col_mod
reload(col_mod)

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

PLATFORM_TARGETS  = {"twitter": 20000, "youtube": 12500, "tiktok": 10000, "instagram": 7500}
COLLECTION_BUFFER = 1.25
TWITTER_SAVE_PATH = DRIVE_ROOT / "data/raw/twitter/raw_twitter.csv"
TWITTER_TARGET    = int(PLATFORM_TARGETS["twitter"] * COLLECTION_BUFFER)

TWITTER_QUERIES = [
    "mantap",         "lucu banget",    "gak nyangka",
    "serius gak sih", "beneran deh",    "ya ampun",
    "literally",      "gaskeun",        "mager banget",
    "rebahan",        "nggak nyangka",  "gilak",
    "astaga",         "receh",          "ngakak",
    "healing",        "gabut",          "gercep",
    "kepo",           "bucin",
]

print("Twitter Collection Config:")
print(f"  Target (with buffer) : {TWITTER_TARGET:,}")
print(f"  Queries              : {len(TWITTER_QUERIES)}")
print(f"  Save path            : {TWITTER_SAVE_PATH}")
print()

# Hapus cache lama agar generate ulang dengan versi yang sudah diperbaiki
if TWITTER_SAVE_PATH.exists():
    TWITTER_SAVE_PATH.unlink()
    print("Cache lama dihapus, regenerating...")

if TWITTER_BEARER_TOKEN:
    collector  = col_mod.TwitterCollector(bearer_token=TWITTER_BEARER_TOKEN)
    df_twitter = collector.collect(queries=TWITTER_QUERIES, target_count=TWITTER_TARGET)
else:
    print("Twitter API key tidak tersedia. Generating mock data...")
    collector  = col_mod.TwitterCollector.__new__(col_mod.TwitterCollector)
    df_twitter = col_mod.TwitterCollector.generate_mock(collector, TWITTER_TARGET)

df_twitter.to_csv(TWITTER_SAVE_PATH, index=False)
print(f"Saved: {len(df_twitter):,} samples -> {TWITTER_SAVE_PATH}")
print(f"\nTwitter dataset shape : {df_twitter.shape}")
print(f"Unique IDs            : {df_twitter['id'].nunique():,}")
print(f"Unique texts          : {df_twitter['text'].nunique():,}")
print(df_twitter.head(3).to_string())

Twitter Collection Config:
  Target (with buffer) : 25,000
  Queries              : 20
  Save path            : /content/drive/MyDrive/EPGT_Research/data/raw/twitter/raw_twitter.csv

Cache lama dihapus, regenerating...
Twitter API key tidak tersedia. Generating mock data...
Saved: 25,000 samples -> /content/drive/MyDrive/EPGT_Research/data/raw/twitter/raw_twitter.csv

Twitter dataset shape : (25000, 4)
Unique IDs            : 25,000
Unique texts          : 20
                                    text                         timestamp              id platform
0   healing dulu yuk 🌿😌 butuh refreshing  2026-03-08T10:00:35.008333+00:00  twitter_000000  twitter
1  gercep dong jangan lama 😤 keburu basi  2026-03-08T10:00:35.008359+00:00  twitter_000001  twitter
2   healing dulu yuk 🌿😌 butuh refreshing  2026-03-08T10:00:35.008365+00:00  twitter_000002  twitter


### Cell 4.2 — YouTube Comments Collection

In [11]:
# ============================================================
# CELL 4.2 — YouTube Comments Collection
# Target: 12,500 samples (25% of total dataset)
# ============================================================

import sys, logging
from pathlib import Path
from importlib import reload
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/EPGT_Research")
sys.path.insert(0, str(DRIVE_ROOT / "src"))

import data.collector as col_mod
reload(col_mod)

YOUTUBE_SAVE_PATH = DRIVE_ROOT / "data/raw/youtube/raw_youtube.csv"
YOUTUBE_TARGET    = int(PLATFORM_TARGETS["youtube"] * COLLECTION_BUFFER)

print("YouTube Collection Config:")
print(f"  Target (with buffer) : {YOUTUBE_TARGET:,}")
print(f"  API Quota            : 10,000 units/day")
print(f"  Save path            : {YOUTUBE_SAVE_PATH}")
print()

if YOUTUBE_SAVE_PATH.exists():
    YOUTUBE_SAVE_PATH.unlink()
    print("Cache lama dihapus, regenerating...")

if YOUTUBE_API_KEY:
    collector  = col_mod.YouTubeCollector(api_key=YOUTUBE_API_KEY)
    df_youtube = collector.collect(target_count=YOUTUBE_TARGET)
else:
    print("YouTube API key tidak tersedia. Generating mock data...")
    collector  = col_mod.YouTubeCollector.__new__(col_mod.YouTubeCollector)
    df_youtube = col_mod.YouTubeCollector.generate_mock(collector, YOUTUBE_TARGET)

df_youtube.to_csv(YOUTUBE_SAVE_PATH, index=False)
print(f"Saved: {len(df_youtube):,} samples -> {YOUTUBE_SAVE_PATH}")
print(f"\nYouTube dataset shape : {df_youtube.shape}")
print(f"Unique IDs            : {df_youtube['id'].nunique():,}")
print(f"Unique texts          : {df_youtube['text'].nunique():,}")
print(df_youtube.head(3).to_string())

YouTube Collection Config:
  Target (with buffer) : 15,625
  API Quota            : 10,000 units/day
  Save path            : /content/drive/MyDrive/EPGT_Research/data/raw/youtube/raw_youtube.csv

Cache lama dihapus, regenerating...
YouTube API key tidak tersedia. Generating mock data...
Saved: 15,625 samples -> /content/drive/MyDrive/EPGT_Research/data/raw/youtube/raw_youtube.csv

YouTube dataset shape : (15625, 4)
Unique IDs            : 15,625
Unique texts          : 20
                                               text                         timestamp              id platform
0   gilak beneran gak nyangka plot twistnya 💀 anjir  2026-03-08T10:00:35.326637+00:00  youtube_000000  youtube
1      receh tapi menghibur 😄 terus berkarya ya kak  2026-03-08T10:00:35.326660+00:00  youtube_000001  youtube
2  konten ini bagus banget sih 👍👍 subscribe dulu ah  2026-03-08T10:00:35.326665+00:00  youtube_000002  youtube


### Cell 4.3 — TikTok Comments Collection

In [12]:
# ============================================================
# CELL 4.3 — TikTok Comments Collection
# Target: 10,000 samples (20% of total dataset)
# ============================================================

import sys, logging
from pathlib import Path
from importlib import reload
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/EPGT_Research")
sys.path.insert(0, str(DRIVE_ROOT / "src"))

import data.collector as col_mod
reload(col_mod)

TIKTOK_SAVE_PATH = DRIVE_ROOT / "data/raw/tiktok/raw_tiktok.csv"
TIKTOK_TARGET    = int(PLATFORM_TARGETS["tiktok"] * COLLECTION_BUFFER)

print("TikTok Collection Config:")
print(f"  Target (with buffer) : {TIKTOK_TARGET:,}")
print(f"  Mode                 : Research API / Playwright / Mock")
print(f"  Save path            : {TIKTOK_SAVE_PATH}")
print()

if TIKTOK_SAVE_PATH.exists():
    TIKTOK_SAVE_PATH.unlink()
    print("Cache lama dihapus, regenerating...")

print("Generating TikTok mock data...")
collector = col_mod.TikTokCollector()
df_tiktok = collector.generate_mock(TIKTOK_TARGET)

df_tiktok.to_csv(TIKTOK_SAVE_PATH, index=False)
print(f"Saved: {len(df_tiktok):,} samples -> {TIKTOK_SAVE_PATH}")
print(f"\nTikTok dataset shape : {df_tiktok.shape}")
print(f"Unique IDs           : {df_tiktok['id'].nunique():,}")
print(f"Unique texts         : {df_tiktok['text'].nunique():,}")
print(df_tiktok.head(3).to_string())

TikTok Collection Config:
  Target (with buffer) : 12,500
  Mode                 : Research API / Playwright / Mock
  Save path            : /content/drive/MyDrive/EPGT_Research/data/raw/tiktok/raw_tiktok.csv

Cache lama dihapus, regenerating...
Generating TikTok mock data...
Saved: 12,500 samples -> /content/drive/MyDrive/EPGT_Research/data/raw/tiktok/raw_tiktok.csv

TikTok dataset shape : (12500, 4)
Unique IDs           : 12,500
Unique texts         : 20
                                text                         timestamp             id platform
0  plot twist terkejut 😲💀 ga nyangka  2026-03-08T10:00:35.646139+00:00  tiktok_000000   tiktok
1    beneran gak ketawa 💀 serius ini  2026-03-08T10:00:35.646174+00:00  tiktok_000001   tiktok
2      ngapain sih 😭😭 ga ada kerjaan  2026-03-08T10:00:35.646179+00:00  tiktok_000002   tiktok


### Cell 4.4 — Instagram Comments Collection

In [13]:
# ============================================================
# CELL 4.4 — Instagram Comments Collection (FIXED)
# Target: 7,500 samples (15% of total dataset)
# ============================================================

import sys, logging
from pathlib import Path
from importlib import reload
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/EPGT_Research")
sys.path.insert(0, str(DRIVE_ROOT / "src"))

import data.collector as col_mod
reload(col_mod)

INSTAGRAM_SAVE_PATH = DRIVE_ROOT / "data/raw/instagram/raw_instagram.csv"
INSTAGRAM_TARGET    = int(PLATFORM_TARGETS["instagram"] * COLLECTION_BUFFER)

print("Instagram Collection Config:")
print(f"  Target (with buffer) : {INSTAGRAM_TARGET:,}")
print(f"  Save path            : {INSTAGRAM_SAVE_PATH}")
print()

if INSTAGRAM_SAVE_PATH.exists():
    INSTAGRAM_SAVE_PATH.unlink()
    print("Cache lama dihapus, regenerating...")

if INSTAGRAM_USERNAME and INSTAGRAM_PASSWORD:
    collector    = col_mod.InstagramCollector(
        username = INSTAGRAM_USERNAME,
        password = INSTAGRAM_PASSWORD,
    )
    df_instagram = collector.collect(target_count=INSTAGRAM_TARGET, use_mock=False)
else:
    print("Instagram credentials tidak tersedia. Generating mock data...")
    collector    = col_mod.InstagramCollector()
    df_instagram = collector.generate_mock(INSTAGRAM_TARGET)

df_instagram.to_csv(INSTAGRAM_SAVE_PATH, index=False)
print(f"Saved: {len(df_instagram):,} samples -> {INSTAGRAM_SAVE_PATH}")
print(f"\nInstagram dataset shape : {df_instagram.shape}")
print(f"Unique IDs              : {df_instagram['id'].nunique():,}")
print(f"Unique texts            : {df_instagram['text'].nunique():,}")
print(df_instagram.head(3).to_string())


Instagram Collection Config:
  Target (with buffer) : 9,375
  Save path            : /content/drive/MyDrive/EPGT_Research/data/raw/instagram/raw_instagram.csv

Cache lama dihapus, regenerating...
Instagram credentials tidak tersedia. Generating mock data...
Saved: 9,375 samples -> /content/drive/MyDrive/EPGT_Research/data/raw/instagram/raw_instagram.csv

Instagram dataset shape : (9375, 4)
Unique IDs              : 9,375
Unique texts            : 20
                                         text                         timestamp         id   platform
0      ini beneran natural? 😲✨ glowing banget  2026-03-08T10:00:35.807026+00:00  ig_000000  instagram
1  resepnya boleh dong share 🍳😋 keliatan enak  2026-03-08T10:00:35.807053+00:00  ig_000001  instagram
2    traveling goals banget 🌏✈️ pengen kesini  2026-03-08T10:00:35.807058+00:00  ig_000002  instagram


---
## BAGIAN 5 — DATA CONSOLIDATION & QUALITY CHECK
### Cell 5.1 — Gabungkan Semua Platform

In [14]:
# ============================================================
# CELL 5.1 — Consolidate All Platforms
# ============================================================

import pandas as pd
import hashlib

# Load semua raw data
dfs = {
    "twitter"  : pd.read_csv(DRIVE_ROOT / "data/raw/twitter/raw_twitter.csv"),
    "youtube"  : pd.read_csv(DRIVE_ROOT / "data/raw/youtube/raw_youtube.csv"),
    "tiktok"   : pd.read_csv(DRIVE_ROOT / "data/raw/tiktok/raw_tiktok.csv"),
    "instagram": pd.read_csv(DRIVE_ROOT / "data/raw/instagram/raw_instagram.csv"),
}

print("Raw collection summary:")
for platform, df in dfs.items():
    pct = len(df) / sum(len(d) for d in dfs.values()) * 100
    print(f"  {platform:<12}: {len(df):>7,} samples  ({pct:.1f}%)")

# Gabungkan
df_all = pd.concat(list(dfs.values()), ignore_index=True)

# Pastikan kolom standar
for col in ["id", "text", "platform", "timestamp"]:
    if col not in df_all.columns:
        df_all[col] = None

print(f"\nTotal before dedup : {len(df_all):,}")

# Cross-platform deduplication
def compute_hash(text: str) -> str:
    return hashlib.md5(str(text).strip().lower().encode()).hexdigest()

df_all["text_hash"] = df_all["text"].apply(compute_hash)
df_all = df_all.drop_duplicates(subset="text_hash").reset_index(drop=True)
df_all = df_all.drop(columns=["text_hash"])

print(f"Total after dedup  : {len(df_all):,}")
print(f"Duplicates removed : {sum(len(d) for d in dfs.values()) - len(df_all):,}")

# Simpan raw consolidated
CONSOLIDATED_PATH = DRIVE_ROOT / "data/raw/raw_consolidated.csv"
df_all.to_csv(CONSOLIDATED_PATH, index=False)
print(f"\nConsolidated saved: {CONSOLIDATED_PATH}")

Raw collection summary:
  twitter     :  25,000 samples  (40.0%)
  youtube     :  15,625 samples  (25.0%)
  tiktok      :  12,500 samples  (20.0%)
  instagram   :   9,375 samples  (15.0%)

Total before dedup : 62,500
Total after dedup  : 80
Duplicates removed : 62,420

Consolidated saved: /content/drive/MyDrive/EPGT_Research/data/raw/raw_consolidated.csv


### Cell 5.2 — Aplikasi Final Inclusion Filter & Quality Check

In [15]:
# ============================================================
# CELL 5.2 — Apply Final Inclusion Filter & Quality Check
# ============================================================

import re
import emoji
from tqdm.auto import tqdm

# Build emoji pattern
emoji_pattern = re.compile(
    "(" + "|".join(
        re.escape(e)
        for e in sorted(emoji.EMOJI_DATA.keys(), key=len, reverse=True)
    ) + ")"
)

def apply_inclusion_filter(df: pd.DataFrame) -> pd.DataFrame:
    """
    Terapkan inclusion criteria blueprint Section 2.1.3:
    1. Minimum 1 emoji
    2. Minimum 3 token setelah emoji dihapus
    3. Teks tidak kosong
    4. Teks tidak terlalu pendek (< 5 karakter)
    """
    passed = []
    reasons_failed = {"no_emoji": 0, "too_short": 0, "empty": 0}

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Filtering"):
        text = str(row.get("text", ""))

        if not text or len(text) < 5:
            reasons_failed["empty"] += 1
            continue

        emoji_found = emoji_pattern.findall(text)
        if len(emoji_found) < MIN_EMOJI_COUNT:
            reasons_failed["no_emoji"] += 1
            continue

        cleaned = emoji_pattern.sub("", text).strip()
        tokens  = cleaned.split()
        if len(tokens) < MIN_TOKEN_COUNT:
            reasons_failed["too_short"] += 1
            continue

        passed.append(row)

    print(f"\nInclusion filter results:")
    print(f"  Input   : {len(df):,}")
    print(f"  Passed  : {len(passed):,} ({len(passed)/len(df)*100:.1f}%)")
    print(f"  Failed  :")
    for reason, count in reasons_failed.items():
        print(f"    {reason:<12}: {count:,}")

    return pd.DataFrame(passed).reset_index(drop=True)


df_filtered = apply_inclusion_filter(df_all)

# ── Emoji density analysis ────────────────────────────────
def count_emoji(text: str) -> int:
    return len(emoji_pattern.findall(str(text)))

df_filtered["emoji_count"] = df_filtered["text"].apply(count_emoji)

single = (df_filtered["emoji_count"] == 1).sum()
few    = ((df_filtered["emoji_count"] >= 2) & (df_filtered["emoji_count"] <= 3)).sum()
many   = (df_filtered["emoji_count"] >= 4).sum()
total  = len(df_filtered)

print(f"\nEmoji density distribution (blueprint target: 35/40/25%):")
print(f"  1 emoji   (target ~35%): {single:>6,}  ({single/total*100:.1f}%)")
print(f"  2-3 emoji (target ~40%): {few:>6,}  ({few/total*100:.1f}%)")
print(f"  4+ emoji  (target ~25%): {many:>6,}  ({many/total*100:.1f}%)")

print(f"\nPlatform distribution:")
for platform, count in df_filtered["platform"].value_counts().items():
    print(f"  {platform:<12}: {count:>6,}  ({count/total*100:.1f}%)")

Filtering:   0%|          | 0/80 [00:00<?, ?it/s]


Inclusion filter results:
  Input   : 80
  Passed  : 80 (100.0%)
  Failed  :
    no_emoji    : 0
    too_short   : 0
    empty       : 0

Emoji density distribution (blueprint target: 35/40/25%):
  1 emoji   (target ~35%):     27  (33.8%)
  2-3 emoji (target ~40%):     53  (66.2%)
  4+ emoji  (target ~25%):      0  (0.0%)

Platform distribution:
  twitter     :     20  (25.0%)
  youtube     :     20  (25.0%)
  tiktok      :     20  (25.0%)
  instagram   :     20  (25.0%)


### Cell 5.3 — Simpan Raw Dataset Final

In [16]:
# ============================================================
# CELL 5.3 — Save Final Raw Dataset
# ============================================================

import uuid
from datetime import datetime, timezone

# Pastikan semua kolom schema dasar ada
df_final = df_filtered.copy()

# Generate ID jika belum ada
if df_final["id"].isna().any():
    mask = df_final["id"].isna()
    df_final.loc[mask, "id"] = [
        f"epgt_{uuid.uuid4().hex[:12]}"
        for _ in range(mask.sum())
    ]

# Tambahkan kolom label (kosong, diisi saat anotasi di NB02)
for label_col in ["label_intensity", "label_sarcasm", "label_emoji_role"]:
    if label_col not in df_final.columns:
        df_final[label_col] = None

# Reorder kolom sesuai dataset schema blueprint Section 2.1.4
schema_cols = [
    "id", "text", "platform", "timestamp",
    "emoji_count",
    "label_intensity", "label_sarcasm", "label_emoji_role",
]
df_final = df_final.reindex(columns=schema_cols, fill_value=None)

# Simpan
RAW_FINAL_PATH = DRIVE_ROOT / "data/raw/raw_final_for_annotation.csv"
df_final.to_csv(RAW_FINAL_PATH, index=False)

print("Final raw dataset saved:")
print(f"  Path    : {RAW_FINAL_PATH}")
print(f"  Samples : {len(df_final):,}")
print(f"  Columns : {list(df_final.columns)}")
print(f"\nSample records:")
print(df_final[["id", "text", "platform", "emoji_count"]].head(5).to_string())

Final raw dataset saved:
  Path    : /content/drive/MyDrive/EPGT_Research/data/raw/raw_final_for_annotation.csv
  Samples : 80
  Columns : ['id', 'text', 'platform', 'timestamp', 'emoji_count', 'label_intensity', 'label_sarcasm', 'label_emoji_role']

Sample records:
               id                                      text platform  emoji_count
0  twitter_000000      healing dulu yuk 🌿😌 butuh refreshing  twitter            2
1  twitter_000001     gercep dong jangan lama 😤 keburu basi  twitter            1
2  twitter_000004  receh bet ini kontennya 😂 tapi lucu juga  twitter            1
3  twitter_000005        ya elah masa gitu 🙄 gak masuk akal  twitter            1
4  twitter_000006    serius gak sih 🤔 masa iya gitu caranya  twitter            1


---
## BAGIAN 6 — NB01 COMPLETION REPORT
### Cell 6.1 — Summary Report

In [17]:
# ============================================================
# CELL 6.1 — NB01 Completion Report
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 60)
print("NB01 COMPLETION REPORT")
print("EPGT Research — Project Setup & Data Collection")
print("=" * 60)

print("\n[1] PROJECT STRUCTURE")
key_dirs = [
    "data/raw", "data/processed", "src/models",
    "checkpoints", "logs", "outputs"
]
for d in key_dirs:
    exists = (DRIVE_ROOT / d).exists()
    print(f"  {'OK' if exists else 'MISSING':>7}  {d}")

print("\n[2] SOURCE CODE FILES")
src_files = [
    "src/config.py",
    "src/data/collector.py",
    "src/models/epgt.py",
    "src/training/trainer.py",
    "src/utils/drive_manager.py",
]
for f in src_files:
    exists = (DRIVE_ROOT / f).exists()
    print(f"  {'OK' if exists else 'MISSING':>7}  {f}")

print("\n[3] RAW DATA COLLECTION")
raw_files = {
    "twitter"  : "data/raw/twitter/raw_twitter.csv",
    "youtube"  : "data/raw/youtube/raw_youtube.csv",
    "tiktok"   : "data/raw/tiktok/raw_tiktok.csv",
    "instagram": "data/raw/instagram/raw_instagram.csv",
}
total_collected = 0
for platform, path in raw_files.items():
    full = DRIVE_ROOT / path
    if full.exists():
        n = len(pd.read_csv(full))
        total_collected += n
        print(f"  {platform:<12}: {n:>7,} samples")
    else:
        print(f"  {platform:<12}: NOT FOUND")

print(f"  {'Total':<12}: {total_collected:>7,} samples")

print("\n[4] FILTERED DATASET")
final_path = DRIVE_ROOT / "data/raw/raw_final_for_annotation.csv"
if final_path.exists():
    df_check = pd.read_csv(final_path)
    print(f"  Samples ready for annotation: {len(df_check):,}")
    print(f"  Blueprint target             : {TOTAL_TARGET:,}")
    sufficient = len(df_check) >= TOTAL_TARGET
    print(f"  Sufficient for annotation?   : {'YES' if sufficient else 'NO — need more collection'}")

print("\n[5] NEXT STEP")
print("  → NB02: Annotation Management & Dataset Split")
print("    Buka NB02 untuk mengelola anotasi tiga-layer")
print("    dan melakukan stratified split 70/15/15.")
print("\n" + "=" * 60)

NB01 COMPLETION REPORT
EPGT Research — Project Setup & Data Collection

[1] PROJECT STRUCTURE
       OK  data/raw
       OK  data/processed
       OK  src/models
       OK  checkpoints
       OK  logs
       OK  outputs

[2] SOURCE CODE FILES
       OK  src/config.py
       OK  src/data/collector.py
       OK  src/models/epgt.py
       OK  src/training/trainer.py
       OK  src/utils/drive_manager.py

[3] RAW DATA COLLECTION
  twitter     :  25,000 samples
  youtube     :  15,625 samples
  tiktok      :  12,500 samples
  instagram   :   9,375 samples
  Total       :  62,500 samples

[4] FILTERED DATASET
  Samples ready for annotation: 80
  Blueprint target             : 50,000
  Sufficient for annotation?   : NO — need more collection

[5] NEXT STEP
  → NB02: Annotation Management & Dataset Split
    Buka NB02 untuk mengelola anotasi tiga-layer
    dan melakukan stratified split 70/15/15.

